<a href="https://colab.research.google.com/github/brunacorreiade/ruf-/blob/main/ruf_2015.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from pathlib import Path
import hashlib
import shutil
import zipfile
import pandas as pd
from IPython.display import display
from google.colab import files

ANO = 2015
FAIXAS = {
    "QT_ING_0_17": "Até 17", "QT_ING_18_24": "18 a 24",
    "QT_ING_25_29": "25 a 29", "QT_ING_30_34": "30 a 34",
    "QT_ING_35_39": "35 a 39", "QT_ING_40_49": "40 a 49",
    "QT_ING_50_59": "50 a 59", "QT_ING_60_MAIS": "60 ou mais",
}
IDADES = list(FAIXAS)
COLS = ["NU_ANO_CENSO", "CO_IES", "TP_REDE", "TP_CATEGORIA_ADMINISTRATIVA",
        "TP_MODALIDADE_ENSINO", "CO_CINE_ROTULO", "QT_ING",
        "QT_ING_DIURNO", "QT_ING_NOTURNO"] + IDADES
CURSOS = {"0113P01": "Pedagogia", "0923S01": "Serviço Social"}
CATEGORIAS = {1: "Pública federal", 2: "Pública estadual", 3: "Pública municipal",
              4: "Privada com fins lucrativos", 5: "Privada sem fins lucrativos",
              6: "Privada particular", 7: "Especial", 8: "Privada comunitária",
              9: "Privada confessional"}
ESPERADO = {2015: (2922400, 784622), 2024: (5010613, 1981219)}


def escolher_zip(ano):
    def contem_ano(caminho):
        try:
            with zipfile.ZipFile(caminho) as pacote:
                return any(n.upper().endswith(f"MICRODADOS_CADASTRO_CURSOS_{ano}.CSV")
                           for n in pacote.namelist())
        except (OSError, zipfile.BadZipFile):
            return False

    candidatos = sorted(Path("/content").glob("*.zip"), key=lambda p: p.stat().st_mtime)
    correspondentes = [p for p in candidatos if contem_ano(p)]
    if not correspondentes:
        print(f"Selecione o ZIP oficial de {ano}. Você pode selecionar os dois anos juntos.")
        files.upload()
        candidatos = sorted(Path("/content").glob("*.zip"), key=lambda p: p.stat().st_mtime)
        correspondentes = [p for p in candidatos if contem_ano(p)]
    if not correspondentes:
        raise FileNotFoundError(f"Não encontrei o cadastro de cursos de {ano} em um ZIP de /content.")
    escolhido = correspondentes[-1]
    print(f"Ano {ano}: lendo {escolhido.name}")
    return escolhido


def resumir(base, grupos):
    if grupos:
        out = base.groupby(grupos, dropna=False, as_index=False).agg(
            total_ingressantes=("QT_ING", "sum"),
            ingressantes_30_mais=("ingressantes_30_mais", "sum"))
    else:
        out = pd.DataFrame([{"total_ingressantes": base.QT_ING.sum(),
                             "ingressantes_30_mais": base.ingressantes_30_mais.sum()}])
    out = out[out.total_ingressantes > 0].copy()
    out["percentual_30_mais"] = out.ingressantes_30_mais / out.total_ingressantes
    return out


zip_local = escolher_zip(ANO)
with zipfile.ZipFile(zip_local) as pacote:
    nomes = pacote.namelist()
    arquivo_cursos = next(n for n in nomes if n.upper().endswith(
        f"MICRODADOS_CADASTRO_CURSOS_{ANO}.CSV"))
    arquivo_ies = next((n for n in nomes if n.upper().endswith(f"IES_{ANO}.CSV")), None)
    if arquivo_ies is None:
        raise FileNotFoundError("Cadastro de instituições ausente do ZIP.")
    with pacote.open(arquivo_cursos) as entrada:
        cabecalho = pd.read_csv(entrada, sep=";", encoding="latin1", nrows=0).columns
    faltantes = sorted(set(COLS) - set(cabecalho))
    if faltantes:
        raise ValueError(f"Colunas ausentes: {faltantes}")
    with pacote.open(arquivo_cursos) as entrada:
        dados = pd.read_csv(entrada, sep=";", encoding="latin1", usecols=COLS,
                            na_values=["."], low_memory=False,
                            dtype={"CO_IES": "string", "CO_CINE_ROTULO": "string"})
    with pacote.open(arquivo_ies) as entrada:
        ies = pd.read_csv(entrada, sep=";", encoding="latin1",
                          usecols=["CO_IES", "NO_IES"], dtype={"CO_IES": "string"})

for c in ["QT_ING", "QT_ING_DIURNO", "QT_ING_NOTURNO"] + IDADES:
    dados[c] = pd.to_numeric(dados[c], errors="coerce").fillna(0).astype("int64")
dados["ingressantes_30_mais"] = dados[IDADES[3:]].sum(axis=1)
dados["modalidade"] = dados.TP_MODALIDADE_ENSINO.map({1: "Presencial", 2: "EaD"})
dados["rede"] = dados.TP_REDE.map({1: "Pública", 2: "Privada"})
dados["categoria"] = dados.TP_CATEGORIA_ADMINISTRATIVA.map(CATEGORIAS)
dados["curso"] = dados.CO_CINE_ROTULO.str.replace('"', "", regex=False).str.strip().map(CURSOS)
if set(dados.NU_ANO_CENSO.unique()) != {ANO}:
    raise ValueError("O ano no cadastro difere do solicitado.")
if dados.loc[dados.QT_ING > 0, ["modalidade", "rede", "categoria"]].isna().any().any():
    raise ValueError("Há ingressantes sem classificação de modalidade, rede ou categoria.")
if ies.CO_IES.duplicated().any():
    raise ValueError("Código de IES duplicado no cadastro.")

total = int(dados.QT_ING.sum())
total_30 = int(dados.ingressantes_30_mais.sum())
presencial = dados[dados.modalidade == "Presencial"]
checagens = pd.DataFrame([
    ("Total de ingressantes", total, ESPERADO[ANO][0]),
    ("Ingressantes 30+", total_30, ESPERADO[ANO][1]),
    ("Linhas com soma das idades diferente do total",
     int((dados[IDADES].sum(axis=1) != dados.QT_ING).sum()), 0),
    ("Linhas presenciais com soma dos turnos diferente do total",
     int(((presencial.QT_ING_DIURNO + presencial.QT_ING_NOTURNO) != presencial.QT_ING).sum()), 0),
], columns=["checagem", "valor", "esperado"])
checagens["status"] = checagens.apply(
    lambda linha: "OK" if linha.valor == linha.esperado else "REVISAR", axis=1)
display(checagens)
if (checagens.status != "OK").any():
    raise ValueError("As checagens falharam; confira os arquivos antes de usar os resultados.")

tabelas = {"checagens": checagens, "resumo_geral": resumir(dados, []),
           "resumo_modalidade": resumir(dados, ["modalidade"]),
           "resumo_rede": resumir(dados, ["rede"]),
           "modalidade_rede": resumir(dados, ["modalidade", "rede"]),
           "categoria_modalidade": resumir(dados, ["categoria", "rede", "modalidade"])}
tabelas["resumo_geral"].insert(0, "ano", ANO)
idade = dados[IDADES].sum().rename_axis("coluna").reset_index(name="ingressantes")
idade["faixa_etaria"] = idade.coluna.map(FAIXAS)
idade["percentual_total"] = idade.ingressantes / total
tabelas["idade_brasil"] = idade[["faixa_etaria", "ingressantes", "percentual_total"]]
for chave in ["resumo_modalidade", "resumo_rede", "modalidade_rede"]:
    tabelas[chave]["participacao_total_30_mais"] = tabelas[chave].ingressantes_30_mais / total_30
tabelas["resumo_modalidade"]["participacao_total"] = (
    tabelas["resumo_modalidade"].total_ingressantes / total)

def abrir_faixas(base, grupos):
    out = base.groupby(grupos, as_index=False)[IDADES].sum().melt(
        id_vars=grupos, var_name="coluna", value_name="ingressantes")
    out["faixa_etaria"] = out.coluna.map(FAIXAS)
    return out[grupos + ["faixa_etaria", "ingressantes"]]

tabelas["faixa_modalidade_rede"] = abrir_faixas(dados, ["modalidade", "rede"])
cursos = dados[dados.curso.notna()].copy()
if set(cursos.curso.unique()) != set(CURSOS.values()):
    raise ValueError("Um dos cursos da pauta está ausente do cadastro.")
tabelas["resumo_cursos"] = resumir(cursos, ["curso"])
tabelas["cursos_modalidade_rede"] = resumir(cursos, ["curso", "modalidade", "rede"])
tabelas["cursos_por_faixa"] = abrir_faixas(cursos, ["curso", "modalidade", "rede"])
cursos = cursos.merge(ies[["CO_IES", "NO_IES"]], on="CO_IES", how="left", validate="m:1")
if cursos.loc[cursos.QT_ING > 0, "NO_IES"].isna().any():
    raise ValueError("Há instituições sem nome no cadastro de IES.")
tabelas["ies_cursos"] = resumir(
    cursos, ["CO_IES", "NO_IES", "curso", "modalidade", "rede"]
).sort_values("ingressantes_30_mais", ascending=False)

turnos = presencial.groupby("rede", as_index=False)[
    ["QT_ING_DIURNO", "QT_ING_NOTURNO"]
].sum().melt(id_vars="rede", var_name="turno", value_name="ingressantes")
turnos.turno = turnos.turno.map({"QT_ING_DIURNO": "Diurno", "QT_ING_NOTURNO": "Noturno"})
turnos["percentual_na_rede"] = turnos.ingressantes / turnos.groupby("rede").ingressantes.transform("sum")
tabelas["turnos_rede_presencial"] = turnos
tabelas["turnos_presenciais"] = turnos.groupby("turno", as_index=False).ingressantes.sum()
tabelas["turnos_presenciais"]["percentual"] = (
    tabelas["turnos_presenciais"].ingressantes / presencial.QT_ING.sum())

hash_zip = hashlib.sha256()
with zip_local.open("rb") as entrada:
    for bloco in iter(lambda: entrada.read(1024 * 1024), b""):
        hash_zip.update(bloco)
tabelas["fonte"] = pd.DataFrame([{
    "ano": ANO, "zip": zip_local.name, "sha256": hash_zip.hexdigest(),
    "cadastro_cursos": arquivo_cursos, "cadastro_ies": arquivo_ies,
    "pagina_inep": "https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/censo-da-educacao-superior"
}])

for nome in ["resumo_geral", "resumo_modalidade", "resumo_rede", "modalidade_rede",
             "resumo_cursos", "cursos_modalidade_rede", "turnos_rede_presencial"]:
    print(f"\n{ANO}: {nome}")
    display(tabelas[nome])

pasta = Path(f"/content/resultados_inep_{ANO}")
pasta.mkdir(exist_ok=True)
with pd.ExcelWriter(pasta / f"resultados_inep_{ANO}.xlsx", engine="openpyxl") as excel:
    for nome, tabela in tabelas.items():
        tabela.to_csv(pasta / f"{nome}_{ANO}.csv", index=False, encoding="utf-8-sig")
        tabela.to_excel(excel, sheet_name=nome[:31], index=False)
saida = shutil.make_archive(str(pasta), "zip", root_dir=pasta)
print(f"\nPronto: resultados de {ANO}.")
files.download(saida)


Ano 2015: lendo microdados_censo_da_educacao_superior_2015 (1).zip


,checagem,valor,esperado,status
0,Total de ingressantes,2922400,2922400,OK
1,Ingressantes 30+,784622,784622,OK
2,Linhas com soma das idades diferente do total,0,0,OK
3,Linhas presenciais com soma dos turnos diferen...,0,0,OK



2015: resumo_geral


,ano,total_ingressantes,ingressantes_30_mais,percentual_30_mais
0,2015,2922400,784622,0.268485



2015: resumo_modalidade


...


2015: turnos_rede_presencial


,rede,turno,ingressantes,percentual_na_rede
0,Privada,Diurno,509348,0.295513
1,Pública,Diurno,307136,0.609110
2,Privada,Noturno,1214256,0.704487
3,Pública,Noturno,197101,0.390890



Pronto: resultados de 2015.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>